# Проект для «Викишоп»

Интернет-магазин «Викишоп» запускает новый сервис. Теперь пользователи могут редактировать и дополнять описания товаров, как в вики-сообществах. То есть клиенты предлагают свои правки и комментируют изменения других. Магазину нужен инструмент, который будет искать токсичные комментарии и отправлять их на модерацию. 

Требуется обучить модель для классификации комментариев на позитивные и негативные. В нашем распоряжении набор данных с разметкой о токсичности правок.

Необходимо построить модель со значением метрики качества *F1* не меньше 0.75. 

**Инструкция по выполнению**

1. Загрузить и подготовить данные.
2. Обучить разные модели. 
3. Сделать выводы.

Для выполнения проекта применять *BERT* необязательно, но вы можете попробовать.

**Описание данных**

Данные находятся в файле `toxic_comments.csv`. Столбец *text* в нём содержит текст комментария, а *toxic* — целевой признак.

## Подготовка

In [11]:
#pip install pandas jupyter pandarallel requests tqdm

     |████████████████████████████████| 115 kB 1.4 MB/s eta 0:00:01
     |████████████████████████████████| 282 kB 4.8 MB/s eta 0:00:01
  Created wheel for pandarallel: filename=pandarallel-1.6.5-py3-none-any.whl size=16679 sha256=f5a5a774ff3cb33552338fe7db738fff9faafe2ad171677d2bb150914e764d93
  Stored in directory: /home/jovyan/.cache/pip/wheels/39/fe/8d/f7912d85b21ea72c9a8a3fefa6141fab92e575a67ab17c4474
Successfully built pandarallel
Note: you may need to restart the kernel to use updated packages.


In [14]:
import pandas as pd 
import numpy as np
import spacy


from sklearn.pipeline import Pipeline, make_pipeline

import re
import nltk
nltk.download('averaged_perceptron_tagger')
from nltk.tokenize import TweetTokenizer
from nltk.stem import WordNetLemmatizer
from nltk.corpus import wordnet, stopwords as nltk_stopwords
from tqdm import tqdm
tqdm.pandas()
from sklearn.feature_extraction.text import TfidfVectorizer, TfidfTransformer

from sklearn.model_selection import train_test_split, RandomizedSearchCV, GridSearchCV

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import LinearSVC
import lightgbm as lgb

from sklearn.metrics import f1_score, accuracy_score, classification_report

from pandarallel import pandarallel
tqdm.pandas(desc="progress")
pandarallel.initialize(progress_bar=True)

import os
import time
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /home/jovyan/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!


In [2]:
#Константа random_state
RS = 42

In [3]:
#Загрузка данных
df = pd.read_csv('/datasets/toxic_comments.csv', index_col='Unnamed: 0')

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 159292 entries, 0 to 159450
Data columns (total 2 columns):
 #   Column  Non-Null Count   Dtype 
---  ------  --------------   ----- 
 0   text    159292 non-null  object
 1   toxic   159292 non-null  int64 
dtypes: int64(1), object(1)
memory usage: 3.6+ MB


In [6]:
df.sample(5)

,text,toxic
115460,"any different views, just say it out \n\nany d...",0
61419,"'Usual argument' here, or more broadly?80.229....",0
85507,I bet Onceinawhile is an Arab.,0
52936,Trigon Engineering Society\nHi Tlmii. If you ...,0
120792,"July 2006 (UTC)\n\nIn reply, please see my com...",0


In [6]:
#Количество пропусков
df.isna().mean()

text     0.0
toxic    0.0
dtype: float64

In [7]:
#Наличие дубликатов
df.duplicated(subset=['text']).sum()

0

In [8]:
#Количество значений по категориям
df['toxic'].value_counts()

0    143106
1     16186
Name: toxic, dtype: int64

In [9]:
#Процентное соотношение категорий
df.groupby('toxic').size() / len(df)

toxic
0    0.898388
1    0.101612
dtype: float64

**Вывод:**

- В датасете отсутствуют пропуски и дубликаты;
- 90% твитов относятся к категории не токсичные.

Функция добавления POS-тега

In [7]:
def wordnet_pos(word):
    '''
    Map POS tag to first character lemmatize() accepts
    '''
    tag = nltk.pos_tag([word])[0][1][0].upper()
    
    tag_dict = {'J': wordnet.ADJ,
                'N': wordnet.NOUN,
                'V': wordnet.VERB,
                'R': wordnet.ADV}
    
    return tag_dict.get(tag, wordnet.NOUN)

Очистим текстовые данные от лишних символов и лематизируем тексты.

In [20]:
lemmatize = nltk.WordNetLemmatizer()

def cleaning(row):
    text = re.sub('[^a-zA-Z]', ' ', row.text).lower() # удаляем неалфавитные символы, приводим к нижнему регистру
    text = nltk.word_tokenize(text, language = 'english') # токенизируем слова
    text = [lemmatize.lemmatize(word, wordnet_pos(word)) for word in text] # лемматирзируем слова
    row['text_new'] = ' '.join(text) # соединяем слова
    return row

df_clear = df.progress_apply(cleaning, axis=1)

100%|██████████| 159292/159292 [22:30<00:00, 117.92it/s]


In [21]:
stopwords = set(nltk_stopwords.words('english'))

Разделим данные на выборки

In [22]:
X = df_clear['text_new']
y = df_clear['toxic']

In [23]:
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=RS, stratify=y)

In [24]:
print(f"Размер обучающего корпуса: {len(X_train)}")
print(f"Размер тренировочного корпуса: {len(X_test)}")

Размер обучающего корпуса: 119469
Размер тренировочного корпуса: 39823


Векторизуем текстовые данные, удалим стоп-слова.

In [25]:
tf_idf_vec = TfidfVectorizer(ngram_range=(1,1), stop_words=stopwords,
               min_df=3, max_df=0.9, strip_accents=None, use_idf=1,
               smooth_idf=1, sublinear_tf=1)

In [19]:
#X_train_vec = tf_idf_vec.fit_transform(X_train)

In [20]:
#X_test_vec = tf_idf_vec.transform(X_test)

In [27]:
print(f"Размер обучающего датасета: {X_train.shape}")
print(f"Размер тренировочного датасета: {X_test.shape}")

Размер обучающего датасета: (119469,)
Размер тренировочного датасета: (39823,)


## Обучение

In [28]:
results = {} # словарь с результатами моделей

In [29]:
pipe_lr = make_pipeline(
        tf_idf_vec,
        LogisticRegression(
        class_weight='balanced',
        random_state=RS,
        n_jobs=-1,
        )
)

In [30]:
param_grid = {
    'logisticregression__solver':['newton-cg', 'lbfgs', 'liblinear', 'sag', 'saga'],
    'logisticregression__C':[.001, .01],
    'logisticregression__max_iter':[500, 1000]
}

gs_lr = RandomizedSearchCV(
    pipe_lr,
    param_grid,
    cv=5,
    scoring='f1',
    n_jobs=-1,
    n_iter = 10
)

In [31]:
%time
gs_lr.fit(X_train, y_train)

None

CPU times: user 2 µs, sys: 0 ns, total: 2 µs
Wall time: 5.25 µs


In [32]:
prediction_lr = gs_lr.predict(X_train)

cr_lr = classification_report(
    y_train, prediction_lr,
    output_dict=True, zero_division=0
)

results['Logistic Regression'] = cr_lr['weighted avg']

pd.DataFrame(cr_lr).round(decimals=3).transpose()

,precision,recall,f1-score,support
0,0.967,0.964,0.966,107329.000
1,0.691,0.711,0.701,12140.000
accuracy,0.938,0.938,0.938,0.938
macro avg,0.829,0.837,0.833,119469.000
weighted avg,0.939,0.938,0.939,119469.000


In [34]:
pipe_rf = make_pipeline(
        tf_idf_vec,
        RandomForestClassifier(
        random_state=RS,
        n_jobs=-1,
        class_weight='balanced',
        verbose=0
    )
)

In [35]:
param_grid = {
    'randomforestclassifier__n_estimators':np.arange(100, 151, 50),
    'randomforestclassifier__max_depth':np.arange(1, 10, 2),
}

gs_rf = RandomizedSearchCV(
    pipe_rf,
    param_grid,
    scoring='f1',
    n_jobs=-1,
    verbose=1
)

In [36]:
%time 
gs_rf.fit(X_train, y_train)

None

CPU times: user 2 µs, sys: 0 ns, total: 2 µs
Wall time: 5.01 µs
Fitting 5 folds for each of 10 candidates, totalling 50 fits


In [37]:
prediction_rf = gs_rf.predict(X_train)

cr_rf = classification_report(
    y_train, prediction_rf,
    output_dict=True, zero_division=0
)

results['Random Forest Classifier'] = cr_rf['weighted avg']

pd.DataFrame(cr_rf).round(decimals=3).transpose()

,precision,recall,f1-score,support
0,0.981,0.710,0.824,107329.000
1,0.256,0.880,0.397,12140.000
accuracy,0.728,0.728,0.728,0.728
macro avg,0.619,0.795,0.610,119469.000
weighted avg,0.908,0.728,0.781,119469.000


In [38]:
pipe_svc = make_pipeline(
        tf_idf_vec,
        LinearSVC(
        class_weight='balanced',
        random_state=RS
    )
)

In [39]:
param_grid = {
    'linearsvc__loss':['hinge', 'squared_hinge'],
    'linearsvc__C':[0.0001, .001, .01]
}

gs_svc = RandomizedSearchCV(
    pipe_svc,
    param_grid,
    scoring='f1',
    n_jobs=-1,
    verbose=1
)

In [40]:
%time 
gs_svc.fit(X_train, y_train)

None

CPU times: user 3 µs, sys: 0 ns, total: 3 µs
Wall time: 7.39 µs
Fitting 5 folds for each of 6 candidates, totalling 30 fits


In [41]:
prediction_svc = gs_svc.predict(X_train)

cr_svc = classification_report(
    y_train, prediction_svc,
    output_dict=True, zero_division=0
)

results['Support Vector Classifier'] = cr_svc['weighted avg']

pd.DataFrame(cr_svc).round(decimals=3).transpose()

,precision,recall,f1-score,support
0,0.976,0.959,0.967,107329.000
1,0.684,0.788,0.732,12140.000
accuracy,0.941,0.941,0.941,0.941
macro avg,0.830,0.873,0.850,119469.000
weighted avg,0.946,0.941,0.943,119469.000


In [42]:
pipe_gbm = make_pipeline(
        tf_idf_vec,
        lgb.LGBMClassifier(
        objective='binary',
        metric='f1',
        n_jobs=-1,
        verbosity=-1,
        random_state=RS
    )
)

In [43]:
param_grid = {
    'lgbmclassifier__max_depth':[-1, 10, 20],
    'lgbmclassifier__num_leaves':[10, 31],
    'lgbmclassifier__learning_rate':[.1, .5]
}
              
gs_gbm = RandomizedSearchCV(
    pipe_gbm,
    param_grid,
    scoring='f1',
    n_jobs=-1,
    verbose=1
)

In [44]:
%time 
gs_gbm.fit(X_train, y_train)

None

CPU times: user 2 µs, sys: 0 ns, total: 2 µs
Wall time: 6.91 µs
Fitting 5 folds for each of 10 candidates, totalling 50 fits


In [45]:
prediction_gbm = gs_gbm.predict(X_train)

cr_gbm = classification_report(
    y_train, prediction_gbm,
    output_dict=True, zero_division=0
)

results['Light GBM'] = cr_gbm['weighted avg']

pd.DataFrame(cr_gbm).round(decimals=3).transpose()

,precision,recall,f1-score,support
0,0.983,0.996,0.990,107329.000
1,0.960,0.850,0.902,12140.000
accuracy,0.981,0.981,0.981,0.981
macro avg,0.972,0.923,0.946,119469.000
weighted avg,0.981,0.981,0.981,119469.000


In [46]:
#Сводная таблицу с результатами.
pd.DataFrame(results).drop(index='support').round(decimals=3).transpose()

,precision,recall,f1-score
Logistic Regression,0.939,0.938,0.939
Random Forest Classifier,0.908,0.728,0.781
Support Vector Classifier,0.946,0.941,0.943
Light GBM,0.981,0.981,0.981


# Проверим лучшую модель бустинга на тестовых данных

In [50]:
prediction_gbm = gs_gbm.predict(X_test)

cr_gbm = classification_report(
    y_test, prediction_gbm,
    output_dict=True, zero_division=0
)

results['Light GBM'] = cr_gbm['weighted avg']

pd.DataFrame(cr_gbm).round(decimals=3).transpose()

,precision,recall,f1-score,support
0,0.967,0.984,0.976,35777.000
1,0.836,0.704,0.765,4046.000
accuracy,0.956,0.956,0.956,0.956
macro avg,0.902,0.844,0.870,39823.000
weighted avg,0.954,0.956,0.954,39823.000


## Выводы

- Данные загружены;
- Выполнены тегирование, токенизация и лематизация
- Текстовая часть векторизована с помощью TfidfVectorizer
- Обучено 4 модели
- Наилучшая метрика F1 получена на модели градиентного бустинга основаного на деревях решений.


С учетом малого количества коментариев категории "1" (токсичных) можно сделать вывод, что будет преобладать пресловутая "ошибка выжившего" и любая модель машинного обучения будет лучше предсказывать категорию класса "0".


## Чек-лист проверки

- [x]  Jupyter Notebook открыт
- [ ]  Весь код выполняется без ошибок
- [ ]  Ячейки с кодом расположены в порядке исполнения
- [ ]  Данные загружены и подготовлены
- [ ]  Модели обучены
- [ ]  Значение метрики *F1* не меньше 0.75
- [ ]  Выводы написаны